In [8]:
from pinecone_datasets import list_datasets

list_datasets()

['ANN_DEEP1B_d96_angular',
 'ANN_Fashion-MNIST_d784_euclidean',
 'ANN_GIST_d960_euclidean',
 'ANN_GloVe_d100_angular',
 'ANN_GloVe_d200_angular',
 'ANN_GloVe_d25_angular',
 'ANN_GloVe_d50_angular',
 'ANN_LastFM_d64_angular',
 'ANN_MNIST_d784_euclidean',
 'ANN_NYTimes_d256_angular',
 'ANN_SIFT1M_d128_euclidean',
 'amazon_toys_quora_all-MiniLM-L6-bm25',
 'cohere-768',
 'it-threat-data-test',
 'it-threat-data-train',
 'langchain-python-docs-text-embedding-ada-002',
 'mnist',
 'movielens-user-ratings',
 'msmarco-v1-bm25-allMiniLML6V2',
 'nq-768-tasb',
 'quora_all-MiniLM-L6-bm25-100K',
 'quora_all-MiniLM-L6-bm25',
 'quora_all-MiniLM-L6-v2_Splade-100K',
 'quora_all-MiniLM-L6-v2_Splade',
 'squad-text-embedding-ada-002',
 'wikipedia-simple-text-embedding-ada-002-100K',
 'wikipedia-simple-text-embedding-ada-002',
 'yfcc-100K-filter-euclidean',
 'yfcc-10M-filter-euclidean',
 'yfcc-10M-filter-euclidean',
 'youtube-transcripts-text-embedding-ada-002']

In [1]:
from pinecone_datasets import load_dataset

dataset = load_dataset("squad-text-embedding-ada-002")
dataset.head()

,id,values,sparse_values,metadata,blob
0,5733be284776f41900661182,"[-0.010262451963272523, 0.02222637996192584, -...",None,"{'text': 'Architecturally, the school has a Ca...",None
1,5733bf84d058e614000b61be,"[-0.009786712423983223, -0.013988726438873078,...",None,"{'text': 'As at most other universities, Notre...",None
2,5733bed24776f41900661188,"[0.013343917696606181, -0.0007001232846109822,...",None,{'text': 'The university is the major seat of ...,None
3,5733a6424776f41900660f51,"[-0.0085222901071539, 0.004399558219521822, -0...",None,{'text': 'The College of Engineering was estab...,None
4,5733a70c4776f41900660f64,"[-0.006695996885869355, -0.02067068565761649, ...",None,{'text': 'All of Notre Dame's undergraduate st...,None


In [2]:
len(dataset)

18891

In [3]:
# we drop sparse_values as they are not needed for this example
dataset.documents.drop(["sparse_values", "blob"], axis=1, inplace=True)

dataset.head()

,id,values,metadata
0,5733be284776f41900661182,"[-0.010262451963272523, 0.02222637996192584, -...","{'text': 'Architecturally, the school has a Ca..."
1,5733bf84d058e614000b61be,"[-0.009786712423983223, -0.013988726438873078,...","{'text': 'As at most other universities, Notre..."
2,5733bed24776f41900661188,"[0.013343917696606181, -0.0007001232846109822,...",{'text': 'The university is the major seat of ...
3,5733a6424776f41900660f51,"[-0.0085222901071539, 0.004399558219521822, -0...",{'text': 'The College of Engineering was estab...
4,5733a70c4776f41900660f64,"[-0.006695996885869355, -0.02067068565761649, ...",{'text': 'All of Notre Dame's undergraduate st...


In [4]:
import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate

    Authenticate()

In [5]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")

# configure client
pc = Pinecone(api_key=api_key)

In [6]:
from pinecone import ServerlessSpec

cloud = os.environ.get("PINECONE_CLOUD") or "aws"
region = os.environ.get("PINECONE_REGION") or "us-east-1"

spec = ServerlessSpec(cloud=cloud, region=region)

In [7]:
index_name = "haozhehw7"

In [20]:
import time

if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

# we create a new index
pc.create_index(
    index_name,
    dimension=1536,  # dimensionality of text-embedding-ada-002
    metric="dotproduct",
    spec=spec,
)

# wait for index to be initialized
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

In [21]:
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

In [22]:
index.upsert_from_dataframe(dataset.documents, batch_size=100)

sending upsert requests:   0%|          | 0/18891 [00:00<?, ?it/s]

{'upserted_count': 18891}

In [23]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 19700}},
 'total_vector_count': 19700}

In [ ]:
!pip install sentence-transformers

In [24]:
from sentence_transformers import SentenceTransformer

# Load the Sentence-Transformer model
model_name = "sangmini/msmarco-cotmae-MiniLM-L12_en-ko-ja"  # This model outputs 1536-dimensional embeddings
model = SentenceTransformer(model_name)

In [25]:
# Example query
query_text = "Who is the current US president?"

# Generate embedding for the query
xq = model.encode(query_text).tolist()

# now query
xc = index.query(vector=xq, top_k=3, include_metadata=True)
xc

{'matches': [{'id': '571a79a64faf5e1900b8a9ba',
              'metadata': {'text': 'Religious Jews have Minhagim, customs, in '
                                   'addition to Halakha, or religious law, and '
                                   'different interpretations of law. '
                                   'Different groups of religious Jews in '
                                   'different geographic areas historically '
                                   'adopted different customs and '
                                   'interpretations. On certain issues, '
                                   'Orthodox Jews are required to follow the '
                                   'customs of their ancestors, and do not '
                                   'believe they have the option of picking '
                                   'and choosing. For this reason, observant '
                                   'Jews at times find it important for '
                                   '

In [26]:
# Function to query Pinecone for each question and retrieve the results
def query_pinecone(question):
    # Generate the embedding for the question
    query_embedding = model.encode(question).tolist()

    # Query Pinecone with the generated embedding
    results = index.query(vector=query_embedding, top_k=3, include_metadata=True)

    # Format and display results
    print(f"Question: {question}")
    print("Top 3 most relevant documents:")
    for idx, match in enumerate(results["matches"]):
        print(f"Document {idx + 1}: {match['id']}")
        print(f"Score: {match['score']}")
        print(f"Text: {match['metadata']['text']}\n")

In [ ]:
# List of questions to query
questions = [
    "If it takes 1 hour for 60 people to play an opera, how many hours will it take 600 people to play the same opera?",
    "Is a pound of feathers or a British pound heavier?",
    "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?",
    "What happens if you crack your knuckles a lot?",
    "If there is a shark in the pool of my basement, is it safe to go upstairs?",
    "How much wood could a woodchuck chuck if there were only 5 pounds of wood in the world?",
    "Who is the current President of the United States?",
    "Was Talos alive?",
    "How many Ls are in the word parallel?",
    "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?",
]

In [28]:
# Loop through all questions and query the Pinecone index
for question in questions:
    query_pinecone(question)

Question: If it takes 1 hour for 60 people to play an opera, how many hours will it take 600 people to play the same opera?
Top 3 most relevant documents:
Document 1: 5706f6a790286e26004fc777
Score: 0.420085341
Text: Originally alphabets were written entirely in majuscule letters, spaced between well-defined upper and lower bounds. When written quickly with a pen, these tended to turn into rounder and much simpler forms. It is from these that the first minuscule hands developed, the half-uncials and cursive minuscule, which no longer stayed bound between a pair of lines. These in turn formed the foundations for the Carolingian minuscule script, developed by Alcuin for use in the court of Charlemagne, which quickly spread across Europe. The advantage of the minuscule over majuscule was improved, faster readability.[citation needed]

Document 2: 572a6ac17a1753140016af22
Score: 0.414906174
Text: Ibn Sīnā refers to the secondary education stage of maktab schooling as a period of specialisa

In [45]:
%%writefile main.py
import os
import time
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from pinecone import ServerlessSpec
from datasets import load_dataset, Dataset
import pandas as pd

# Mock dataset creation if real dataset fails
def create_mock_dataset():
    """Mock a simple dataset for testing."""
    data = {
        'id': [1, 2, 3],
        'text': [
            "What is the capital of France?",
            "Who is the president of the United States?",
            "What is the boiling point of water?"
        ]
    }
    return Dataset.from_dict(data)

# Try to load the actual dataset or fall back to a mock dataset
try:
    dataset = load_dataset("squad-text-embedding-ada-002")
    dataset['documents'].drop(['sparse_values', 'blob'], axis=1, inplace=True)
except Exception as e:
    print(f"Failed to load dataset: {e}. Using mock dataset instead.")
    dataset = create_mock_dataset()

# Authenticate Pinecone if necessary
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# Get the API key from environment variables
api_key = os.environ.get("PINECONE_API_KEY")
if not api_key:
    raise EnvironmentError("PINECONE_API_KEY is required")

# Configure the Pinecone client
pc = Pinecone(api_key=api_key)

# Set cloud and region options
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

# Define the serverless spec for the index
spec = ServerlessSpec(cloud=cloud, region=region)

# Set the index name
index_name = 'haozhehw7'

# If the index exists, delete it
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

# Create a new index
pc.create_index(
    index_name,
    dimension=1536,  # Dimensionality of text-embedding-ada-002
    metric='cosine',
    spec=spec
)

# Wait for the index to be initialized
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

# Connect to the index
index = pc.Index(index_name)

# Upsert the data from the dataset to the index
for batch in tqdm(dataset['documents'].iter_documents(batch_size=100), total=len(dataset['documents']) // 100):
    index.upsert(batch)

# Load the model for querying
model_name = 'sangmini/msmarco-cotmae-MiniLM-L12_en-ko-ja'  # This model outputs 1536-dimensional embeddings
model = SentenceTransformer(model_name)

# Example query
query_text = "Who is the current US president?"

# Generate the embedding for the query
xq = model.encode(query_text).tolist()

# Query the Pinecone index
xc = index.query(vector=xq, top_k=3, include_metadata=True)

# Print the query results
print(xc)



Overwriting main.py


In [50]:
%%writefile test_vectordb.py
import os
import pytest
from sentence_transformers import SentenceTransformer
from unittest.mock import MagicMock
from datasets import Dataset
import time

# Mock Pinecone client and index
@pytest.fixture(scope='module')
def mock_pinecone():
    """Fixture to mock the Pinecone client and index."""

    # Mock the Pinecone Index object
    mock_index = MagicMock()

    # Mock describe_index_stats to return empty data initially
    mock_index.describe_index_stats.return_value = {'total_vector_count': 0}

    # Mock upsert method
    mock_index.upsert.return_value = None

    # Mock query method to return fake results
    mock_index.query.return_value = {
        'matches': [{'id': '1', 'score': 0.98, 'metadata': {'text': 'Example match 1'}},
                    {'id': '2', 'score': 0.96, 'metadata': {'text': 'Example match 2'}}]
    }

    yield mock_index

# Mock dataset
@pytest.fixture
def mock_dataset():
    """Fixture to mock the dataset."""
    data = {
        'id': [1, 2, 3],
        'text': [
            "What is the capital of France?",
            "Who is the president of the United States?",
            "What is the boiling point of water?"
        ]
    }
    return Dataset.from_dict(data)

def test_index_creation(mock_pinecone):
    """Test if the index is created successfully."""
    index = mock_pinecone
    stats = index.describe_index_stats()
    assert stats['total_vector_count'] == 0  # Ensure no vectors are added yet

def test_upsert_data(mock_pinecone, mock_dataset):
    """Test if data can be upserted into the Pinecone index."""
    index = mock_pinecone

    # Convert dataset into batches manually for upserting
    batch_size = 1  # Change the batch size to suit your needs
    for i in range(0, len(mock_dataset), batch_size):
        batch = mock_dataset[i:i+batch_size]  # Slice the dataset into batches
        index.upsert(batch)

    # Since this is a mock test, the total vector count should still be 0
    stats = index.describe_index_stats()
    assert stats['total_vector_count'] == 0  # Mock data isn't actually inserted in this test


def test_query_data(mock_pinecone):
    """Test if querying works with a sample text."""
    index = mock_pinecone
    model_name = 'sangmini/msmarco-cotmae-MiniLM-L12_en-ko-ja'
    model = SentenceTransformer(model_name)

    # Example query
    query_text = "Who is the current US president?"
    query_embedding = model.encode(query_text).tolist()

    # Perform the query using the mock index
    results = index.query(vector=query_embedding, top_k=3, include_metadata=True)

    # Validate that query returns matches
    assert len(results['matches']) > 0  # Ensure we get some matches
    assert results['matches'][0]['score'] > 0.9  # Example to check match score
    assert 'metadata' in results['matches'][0]  # Ensure metadata is present



Overwriting test_vectordb.py


In [51]:
!pytest test_vectordb.py

======================================= test session starts ========================================
platform linux -- Python 3.11.11, pytest-8.3.5, pluggy-1.5.0
rootdir: /content
plugins: typeguard-4.4.2, anyio-3.7.1
collected 3 items                                                                                  

test_vectordb.py ...                                                                         [100%]

======================================== 3 passed in 15.01s ========================================
